In [ ]:
# %pip install scikit-learn
# %pip install matplotlib

En este punto se hará los siguiente:
- División
- Tokenización
- Padding/Truncation
- Balanceo de los train


# División (80train(65train/15val)/20test)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

Cargar las bases de datos

In [ ]:
# Rutas de los datasets
ruta_DATD = r"./Bases de Datos Limpias/1 TheDATD/DATD_limpio.csv"
ruta_SDCNL = r"./Bases de Datos Limpias/2 SDCNL/SDCNL_limpio.csv"
ruta_DU = r"./Bases de Datos Limpias/4 Dataset_Unificado/Dataset_Unificado.csv"

# Cargar datasets
df_DATD = pd.read_csv(ruta_DATD)
df_SDCNL = pd.read_csv(ruta_SDCNL)
df_DU = pd.read_csv(ruta_DU)

# Comprobar tamaños
print("DATD:", df_DATD.shape)
print("SDCNL:", df_SDCNL.shape)
print("DU:", df_DU.shape)

Función para dividir cualquier dataset

In [ ]:
def dividir_dataset(df, nombre_dataset, random_state=42):
    
    print(f"\n--- División de {nombre_dataset} ---")
    
    # Primera división: 80% train_val y 20% test
    train_val, test = train_test_split(
        df,
        test_size=0.20,
        random_state=random_state,
        stratify=df["Label"]
    )
    
    # Segunda división:
    # del 80%, separamos 18.75% para validación
    # 0.1875 * 80% = 15% del total
    train, val = train_test_split(
        train_val,
        test_size=0.1875,
        random_state=random_state,
        stratify=train_val["Label"]
    )
    
    print("Train:", train.shape)
    print("Validation:", val.shape)
    print("Test:", test.shape)
    
    print("\nDistribución de clases en Train:")
    print(train["Label"].value_counts(normalize=True))
    
    print("\nDistribución de clases en Validation:")
    print(val["Label"].value_counts(normalize=True))
    
    print("\nDistribución de clases en Test:")
    print(test["Label"].value_counts(normalize=True))
    
    return train, val, test

Aplicar la división a las tres bases de datos

In [ ]:
train_DATD, val_DATD, test_DATD = dividir_dataset(df_DATD, "DATD")

train_SDCNL, val_SDCNL, test_SDCNL = dividir_dataset(df_SDCNL, "SDCNL")

train_DU, val_DU, test_DU = dividir_dataset(df_DU, "Dataset_Unificado")

In [ ]:
# Carpeta de salida
carpeta_salida = r"./Bases de Datos Splits/"

# Guardar DATD
train_DATD.to_csv(carpeta_salida + "train_DATD.csv", index=False, encoding="utf-8-sig")
val_DATD.to_csv(carpeta_salida + "val_DATD.csv", index=False, encoding="utf-8-sig")
test_DATD.to_csv(carpeta_salida + "test_DATD.csv", index=False, encoding="utf-8-sig")

# Guardar SDCNL
train_SDCNL.to_csv(carpeta_salida + "train_SDCNL.csv", index=False, encoding="utf-8-sig")
val_SDCNL.to_csv(carpeta_salida + "val_SDCNL.csv", index=False, encoding="utf-8-sig")
test_SDCNL.to_csv(carpeta_salida + "test_SDCNL.csv", index=False, encoding="utf-8-sig")

# Guardar DU
train_DU.to_csv(carpeta_salida + "train_DU.csv", index=False, encoding="utf-8-sig")
val_DU.to_csv(carpeta_salida + "val_DU.csv", index=False, encoding="utf-8-sig")
test_DU.to_csv(carpeta_salida + "test_DU.csv", index=False, encoding="utf-8-sig")

print("Splits guardados correctamente.")

# Balanceo de clases -> se va a hacer oversampling a través de Backtranslation

In [ ]:
# =====================================================
# ANALIZAR DISTRIBUCIÓN DE CLASES EN LOS TRAIN
# =====================================================

print("========== DATD ==========")
print(train_DATD["Label"].value_counts().sort_index())

print("\n========== SDCNL ==========")
print(train_SDCNL["Label"].value_counts().sort_index())

print("\n========== DU ==========")
print(train_DU["Label"].value_counts().sort_index())

print("\n========== PORCENTAJES DATD ==========")
print(train_DATD["Label"].value_counts(normalize=True).sort_index()*100)

print("\n========== PORCENTAJES SDCNL ==========")
print(train_SDCNL["Label"].value_counts(normalize=True).sort_index()*100)

print("\n========== PORCENTAJES DU ==========")
print(train_DU["Label"].value_counts(normalize=True).sort_index()*100)

In [ ]:
# =====================================================
# OBJETIVOS DEL OVERSAMPLING
# =====================================================

# SDCNL ya está balanceado, por lo que no se aplica oversampling.

# DATD:
# Clase 1 tiene 356 instancias.
# Cada texto puede generar tres variantes:
# Inglés -> Alemán -> Inglés
# Inglés -> Francés -> Inglés
# Inglés -> Español -> Inglés
#
# 356 originales + (356 * 3 variantes) = 1424
OBJETIVO_DATD = 1424


# DU:
# Clase 2 es la más pequeña, con 593 instancias.
# Se utiliza esta clase para definir un objetivo común
# para las clases 1 y 2.
#
# 593 originales + (593 * 3 variantes) = 2372
OBJETIVO_DU = 2372

print("Objetivo DATD clase 1:", OBJETIVO_DATD)
print("Objetivo DU clases 1 y 2:", OBJETIVO_DU)

In [ ]:
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer

# GPU.
# Comprobar si hay una GPU disponible y seleccionarla
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Usando:', device)

In [ ]:
# =========================================================
# MODELOS HELSINKI PARA RETROTRADUCCION O BACKTRANSLATION
# =========================================================

# Inglés -> Alemán
tokenizer_en_de = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
model_en_de = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-de").to(device)

# Alemán -> Inglés
tokenizer_de_en = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-de-en")
model_de_en = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-de-en").to(device)


# Inglés -> Francés
tokenizer_en_fr = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
model_en_fr = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-fr").to(device)

# Francés -> Inglés
tokenizer_fr_en = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")
model_fr_en = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-fr-en").to(device)


# Inglés -> Español
tokenizer_en_es = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es")
model_en_es = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-es").to(device)

# Español -> Inglés
tokenizer_es_en = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-es-en")
model_es_en = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-es-en").to(device)

print("Modelos de traducción cargados correctamente.")

In [ ]:
# Función de Traducción

def traducir(texto, tokenizer, modelo):
    """
    Traduce un texto utilizando el tokenizer y modelo indicados.
    """

    inputs = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        salida = modelo.generate(**inputs)

    texto_traducido = tokenizer.decode(
        salida[0],
        skip_special_tokens=True
    )

    return texto_traducido

In [ ]:
# Las tres backtranslations

def backtranslation_aleman(texto):
    """Inglés -> Alemán -> Inglés"""
    texto_intermedio = traducir(
        texto,
        tokenizer_en_de,
        model_en_de)
    texto_final = traducir(
        texto_intermedio,
        tokenizer_de_en,
        model_de_en)
    return texto_final


def backtranslation_frances(texto):
    """Inglés -> Francés -> Inglés"""
    texto_intermedio = traducir(
        texto,
        tokenizer_en_fr,
        model_en_fr)
    texto_final = traducir(
        texto_intermedio,
        tokenizer_fr_en,
        model_fr_en)
    return texto_final


def backtranslation_espanol(texto):
    """Inglés -> Español -> Inglés"""
    texto_intermedio = traducir(
        texto,
        tokenizer_en_es,
        model_en_es)
    texto_final = traducir(
        texto_intermedio,
        tokenizer_es_en,
        model_es_en)
    return texto_final

In [ ]:
# Función de oversampling
def aumentar_clase(df_train, clase, objetivo, nombre_dataset):

    # Seleccionar únicamente las instancias de la clase
    # que queremos aumentar
    df_clase = df_train[df_train["Label"] == clase].copy()

    n_actual = len(df_clase)
    n_nuevos = objetivo - n_actual

    print("\n========================================")
    print(f"{nombre_dataset} - Clase {clase}")
    print("========================================")
    print("Instancias actuales:", n_actual)
    print("Objetivo:", objetivo)
    print("Nuevas instancias necesarias:", n_nuevos)

    nuevas_filas = []

    # Tres rutas diferentes de backtranslation
    traducciones = [
        ("Alemán", backtranslation_aleman),
        ("Francés", backtranslation_frances),
        ("Español", backtranslation_espanol)
    ]

    contador = 0

    # Recorrer cada idioma
    for idioma, funcion_backtranslation in traducciones:

        print(f"\nComenzando traducciones vía {idioma}...")

        # Recorrer los textos originales de la clase
        for _, fila in df_clase.iterrows():

            # Parar cuando se alcance el objetivo
            if contador >= n_nuevos:
                break

            nuevo_texto = funcion_backtranslation(
                fila["Text"]
            )

            # Copiar toda la fila original para conservar
            # Label, source y el resto de columnas
            nueva_fila = fila.copy()

            # Sustituir solamente el texto
            nueva_fila["Text"] = nuevo_texto

            nuevas_filas.append(nueva_fila)

            contador += 1

            # Mostrar progreso cada 50 traducciones
            if contador % 50 == 0 or contador == 1:
                print(f"Generadas {contador} de {n_nuevos}")

        if contador >= n_nuevos:
            break

    # Convertir las nuevas instancias en DataFrame
    df_nuevos = pd.DataFrame(nuevas_filas)

    # -------------------------------------------------
    # ELIMINAR DUPLICADOS SOLO DE LOS TEXTOS GENERADOS
    # -------------------------------------------------

    # Eliminar duplicados entre los nuevos textos
    df_nuevos = df_nuevos.drop_duplicates(
        subset=["Text"]
    )

    # Eliminar nuevas traducciones que sean exactamente
    # iguales a algún texto que ya existía en el train.
    # Los textos originales NO se eliminan.
    df_nuevos = df_nuevos[
        ~df_nuevos["Text"].isin(df_train["Text"])
    ]

    # Unir originales + nuevas instancias
    df_balanceado = pd.concat(
        [df_train, df_nuevos],
        ignore_index=True
    )

    # Mezclar las instancias
    df_balanceado = df_balanceado.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    print("\nNuevas instancias válidas añadidas:", len(df_nuevos))

    print("\nDistribución después del oversampling:")
    print(
        df_balanceado["Label"]
        .value_counts()
        .sort_index()
    )

    return df_balanceado

In [ ]:
# =====================================================
# APLICAR OVERSAMPLING
# =====================================================

# DATD:
# Aumentar la clase 1 mediante tres rutas de backtranslation
train_DATD_bal = aumentar_clase(
    train_DATD,
    clase=1,
    objetivo=OBJETIVO_DATD,
    nombre_dataset="DATD"
)


# SDCNL:
# No necesita oversampling porque ya está balanceado
print("\n========================================")
print("SDCNL ya está balanceado.")
print("No se aplica oversampling.")
print("========================================")

train_SDCNL_bal = train_SDCNL.copy()


# DU:
# Aumentar clase 1
train_DU_bal = aumentar_clase(
    train_DU,
    clase=1,
    objetivo=OBJETIVO_DU,
    nombre_dataset="DU"
)

# Aumentar clase 2
train_DU_bal = aumentar_clase(
    train_DU_bal,
    clase=2,
    objetivo=OBJETIVO_DU,
    nombre_dataset="DU"
)

print("\n========================================")
print("Oversampling finalizado.")
print("========================================")

In [ ]:
# =====================================================
# COMPROBAR RESULTADO DEL OVERSAMPLING
# =====================================================

print("========== DATD DESPUÉS DEL OVERSAMPLING ==========")
print(train_DATD_bal["Label"].value_counts().sort_index())

print("\n========== SDCNL ==========")
print(train_SDCNL_bal["Label"].value_counts().sort_index())

print("\n========== DU DESPUÉS DEL OVERSAMPLING ==========")
print(train_DU_bal["Label"].value_counts().sort_index())

In [ ]:
# =====================================================
# UNDERSAMPLING PARCIAL
# =====================================================
# Después del oversampling todavía existe cierto
# desbalance entre las clases.
#
# Se elimina únicamente el 50% de la diferencia restante
# para reducir el desbalance sin perder demasiada
# información de la clase mayoritaria.


# =====================================================
# DATD
# =====================================================

# Número actual de instancias
n_DATD_0 = len(train_DATD_bal[train_DATD_bal["Label"] == 0])
n_DATD_1 = len(train_DATD_bal[train_DATD_bal["Label"] == 1])

# Calcular la diferencia entre ambas clases
diferencia_DATD = n_DATD_0 - n_DATD_1

# Reducir solamente el 50% de esa diferencia
objetivo_DATD_0 = round(
    n_DATD_0 - diferencia_DATD * 0.5)

print("========== UNDERSAMPLING DATD ==========")
print("Clase 0 antes:", n_DATD_0)
print("Clase 1:", n_DATD_1)
print("Clase 0 después:", objetivo_DATD_0)


# Seleccionar aleatoriamente las instancias de clase 0
# que se conservarán
DATD_0 = train_DATD_bal[
    train_DATD_bal["Label"] == 0].sample(
    n=objetivo_DATD_0,
    random_state=42)

# Mantener todas las instancias de clase 1
DATD_1 = train_DATD_bal[
    train_DATD_bal["Label"] == 1]

# Sobrescribir el train balanceado
train_DATD_bal = pd.concat(
    [DATD_0, DATD_1],
    ignore_index=True)


# =====================================================
# DU
# =====================================================

n_DU_0 = len(train_DU_bal[train_DU_bal["Label"] == 0])
n_DU_1 = len(train_DU_bal[train_DU_bal["Label"] == 1])
n_DU_2 = len(train_DU_bal[train_DU_bal["Label"] == 2])

# Como las clases 1 y 2 están prácticamente igualadas,
# utilizar la menor de ambas como referencia
n_DU_min = min(n_DU_1, n_DU_2)

# Calcular la diferencia restante
diferencia_DU = n_DU_0 - n_DU_min

# Eliminar solamente el 50% de esa diferencia
objetivo_DU_0 = round(n_DU_0 - diferencia_DU * 0.5)

print("\n========== UNDERSAMPLING DU ==========")
print("Clase 0 antes:", n_DU_0)
print("Clase 1:", n_DU_1)
print("Clase 2:", n_DU_2)
print("Clase 0 después:", objetivo_DU_0)


# Seleccionar las instancias de clase 0 que conservamos
DU_0 = train_DU_bal[
    train_DU_bal["Label"] == 0].sample(
    n=objetivo_DU_0,
    random_state=42)

# Las clases 1 y 2 se mantienen completas
DU_1 = train_DU_bal[
    train_DU_bal["Label"] == 1]

DU_2 = train_DU_bal[
    train_DU_bal["Label"] == 2]

# Sobrescribir el train balanceado
train_DU_bal = pd.concat(
    [DU_0, DU_1, DU_2],
    ignore_index=True)

In [ ]:
# =====================================================
# MEZCLAR Y COMPROBAR LOS TRAIN BALANCEADOS
# =====================================================

# Mezclar después del undersampling
train_DATD_bal = train_DATD_bal.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

train_DU_bal = train_DU_bal.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


print("========== DATD BALANCEADO ==========")
print(train_DATD_bal["Label"].value_counts().sort_index())

print("\n========== SDCNL BALANCEADO ==========")
print(train_SDCNL_bal["Label"].value_counts().sort_index())

print("\n========== DU BALANCEADO ==========")
print(train_DU_bal["Label"].value_counts().sort_index())

In [ ]:
# =====================================================
# GUARDAR LOS TRAIN BALANCEADOS
# =====================================================

carpeta_splits = r"./Bases de Datos Splits/"

train_DATD_bal.to_csv(carpeta_splits + "train_DATD_balanceado.csv", index=False, encoding="utf-8-sig")
train_SDCNL_bal.to_csv(carpeta_splits + "train_SDCNL_balanceado.csv", index=False, encoding="utf-8-sig")
train_DU_bal.to_csv(carpeta_splits + "train_DU_balanceado.csv", index=False, encoding="utf-8-sig")

print("Train balanceados guardados correctamente.")